# Exploratory Data Analysis: Insurance Claims Fraud

This notebook explores the Kaggle *Insurance Claims Fraud Detection* dataset to prepare it for building the **Intelligent Claims Triage System**. We focus on class balance, missing values, and identifying key predictive features.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## 1. Load and Inspect Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('data/insurance_claims.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Target Class Balance

Let's check the distribution of our target variable `fraud_reported`.

In [ ]:
class_counts = df['fraud_reported'].value_counts()
class_pct = df['fraud_reported'].value_counts(normalize=True) * 100

print("Class Counts:\n", class_counts)
print("\nClass Percentages:\n", class_pct)

fig = px.bar(
    x=class_counts.index,
    y=class_counts.values,
    labels={'x': 'Fraud Reported', 'y': 'Count'},
    title='Target Variable Distribution (fraud_reported)',
    color=class_counts.index,
    color_discrete_sequence=['#3b82f6', '#ef4444']
)
fig.show()

## 3. Missing Values and Placeholders

Checking both standard NaN values and `'?'` placeholders.

In [ ]:
# Identify standard missing values
print("Standard missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Check for '?' values in categorical columns
print("\n'?' placeholder counts:")
for col in df.columns:
    q_count = (df[col] == '?').sum()
    if q_count > 0:
        print(f"{col}: {q_count} ({q_count / len(df) * 100:.2f}%)")

## 4. Key Predictive Feature Distributions

Let's explore the relationships between some top features and the target variable.

In [ ]:
# Incident Severity vs Fraud Reported
fig_severity = px.histogram(
    df, 
    x='incident_severity', 
    color='fraud_reported',
    barmode='group',
    title='Incident Severity by Fraud Status',
    color_discrete_sequence=['#3b82f6', '#ef4444']
)
fig_severity.show()

# Total Claim Amount Distribution
fig_claim = px.box(
    df, 
    x='fraud_reported', 
    y='total_claim_amount',
    color='fraud_reported',
    title='Total Claim Amount Distribution by Fraud Status',
    color_discrete_sequence=['#3b82f6', '#ef4444']
)
fig_claim.show()

## 5. Feature Selection for Pipeline

Based on initial feature importance and business domain, we select the following **10 features** for our model pipeline:

1. `incident_severity` (Categorical: Major Damage, Minor Damage, Total Loss, Trivial Damage)
2. `insured_hobbies` (Categorical: education-level hobbies, base-jumping, chess, crossfit, etc.)
3. `total_claim_amount` (Numerical)
4. `vehicle_claim` (Numerical)
5. `property_claim` (Numerical)
6. `injury_claim` (Numerical)
7. `policy_annual_premium` (Numerical)
8. `months_as_customer` (Numerical)
9. `age` (Numerical)
10. `incident_hour_of_the_day` (Numerical)

We will exclude high-cardinality/ID columns like `policy_bind_date`, `policy_number`, `insured_zip`, and `incident_location` to prevent overfitting.